# Kalman Filter & Sensor Fusion
## Optimal State Estimation for Robotic Systems

This notebook provides a complete, from-scratch implementation of the **Kalman Filter** and its nonlinear extension, the **Extended Kalman Filter (EKF)** — the foundational algorithms for state estimation in robotics.

**What you'll learn:**
1. How Bayesian estimation leads naturally to the Kalman Filter
2. The predict-update cycle and its mathematical derivation
3. Implementation from scratch: 1D tracking, 2D localization, nonlinear estimation
4. Innovation sequence analysis for filter consistency verification
5. Extended Kalman Filter for nonlinear systems (unicycle model)
6. Sensor fusion: combining IMU and GPS for robust estimation

**Prerequisites:** Probability theory (Gaussian distributions, Bayes' theorem), linear algebra (matrix operations, eigendecomposition), basic control theory.

**References:**
- Thrun, Burgard & Fox, *Probabilistic Robotics*, MIT Press, 2005.
- Simon, *Optimal State Estimation: Kalman, H-infinity, and Nonlinear Approaches*, Wiley, 2006.
- Bar-Shalom, Li & Kirubarajan, *Estimation with Applications to Tracking and Navigation*, Wiley, 2001.

---
## 1. The State Estimation Problem

In robotics, we never observe the true state of a system directly. Every sensor is **noisy**, every actuator is **imprecise**, and the world is full of **uncertainty**.

| Sensor | What it measures | Typical noise |
|--------|-----------------|---------------|
| GPS | Position | $\sigma \sim 1{-}5$ m |
| IMU (accelerometer) | Linear acceleration | Bias drift + white noise |
| IMU (gyroscope) | Angular velocity | Bias drift + white noise |
| Wheel encoders | Odometry (velocity) | Slip, discretization |
| LiDAR | Range to obstacles | $\sigma \sim 1{-}3$ cm |
| Camera | Pixel coordinates | Quantization, distortion |

The **state estimation problem** is: given a sequence of noisy sensor measurements $z_1, z_2, \ldots, z_k$ and control inputs $u_1, u_2, \ldots, u_k$, compute the best estimate of the system's true state $x_k$.

The **Kalman Filter** (KF) provides the **optimal** solution to this problem when:
1. The system dynamics are **linear**
2. All noise is **Gaussian**
3. The initial state is **Gaussian**

Under these conditions, the KF computes the exact posterior distribution $P(x_k | z_{1:k}, u_{1:k})$, which is itself Gaussian. No other estimator can achieve lower mean-squared error.

---
## 2. Bayesian Estimation Foundation

The Kalman Filter is a special case of **recursive Bayesian filtering**. To understand it, we start with Bayes' theorem.

### Bayes' Theorem for State Estimation

Given a measurement $z$ and a state $x$:

$$P(x | z) = \frac{P(z | x) \, P(x)}{P(z)} \propto P(z | x) \, P(x)$$

| Term | Name | Meaning |
|------|------|---------|
| $P(x)$ | Prior | Our belief about $x$ before seeing $z$ |
| $P(z \mid x)$ | Likelihood | How likely is measurement $z$ given state $x$ |
| $P(x \mid z)$ | Posterior | Updated belief about $x$ after seeing $z$ |

### Gaussian Case: Precision-Weighted Fusion

Suppose both prior and likelihood are Gaussian:
- Prior: $P(x) = \mathcal{N}(\mu_{\text{prior}}, \sigma^2_{\text{prior}})$
- Likelihood: $P(z | x) = \mathcal{N}(z, \sigma^2_{\text{meas}})$ (i.e., $z = x + v$, $v \sim \mathcal{N}(0, \sigma^2_{\text{meas}})$)

Then the posterior is also Gaussian: $P(x | z) = \mathcal{N}(\mu_{\text{post}}, \sigma^2_{\text{post}})$, where:

$$\boxed{\frac{1}{\sigma^2_{\text{post}}} = \frac{1}{\sigma^2_{\text{prior}}} + \frac{1}{\sigma^2_{\text{meas}}}}$$

$$\boxed{\mu_{\text{post}} = \sigma^2_{\text{post}} \left( \frac{\mu_{\text{prior}}}{\sigma^2_{\text{prior}}} + \frac{z}{\sigma^2_{\text{meas}}} \right)}$$

This is **precision-weighted averaging**: the posterior mean is a weighted average of the prior mean and the measurement, with weights proportional to precision ($1/\sigma^2$). The more precise source gets more weight.

### From Bayes to Kalman

The Kalman Filter emerges when we apply Bayes' theorem **recursively** to a linear-Gaussian dynamical system:
1. **Predict step** (time update): propagate the prior through the dynamics model $\rightarrow$ new prior
2. **Update step** (measurement update): fuse the predicted prior with the new measurement $\rightarrow$ posterior

The posterior at time $k$ becomes the prior at time $k+1$, and the cycle repeats.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from scipy.stats import norm, chi2

%matplotlib inline

plt.rcParams.update({
    'figure.figsize': (12, 5),
    'font.size': 12,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'lines.linewidth': 2,
})

np.random.seed(42)

In [ ]:
# =============================================================================
# Global Constants
# =============================================================================

# General
GRAVITY = 9.81              # m/s^2
FD_EPSILON = 1e-7           # Finite difference step for Jacobian verification

# 1D Falling object (Section 4)
DT_FALL = 0.1               # Time step (s)
T_FALL = 10.0               # Total simulation time (s)
R_FALL = 25.0               # Measurement noise variance (sigma^2 = 25 -> sigma = 5m)
Q_FALL_POS = 0.01           # Process noise position variance
Q_FALL_VEL = 0.1            # Process noise velocity variance
X0_FALL = 100.0             # Initial height (m)
V0_FALL = 0.0               # Initial velocity (m/s)

# 2D Robot Localization (Section 6)
RADIUS_PATH = 50.0          # Circular path radius (m)
PERIOD_PATH = 60.0          # Circular path period (s)
DT_LOC = 0.1                # Time step (s)
T_LOC = 60.0                # Total simulation time (s)
SIGMA_GPS = 5.0             # GPS noise std (m)
SIGMA_ODOM = 0.5            # Odometry noise std (m/s)
GPS_INTERVAL = 10           # GPS update every N steps

# EKF Unicycle (Section 7)
DT_UNI = 0.1                # Time step (s)
T_UNI = 20.0                # Total simulation time (s)
SIGMA_RANGE = 0.5           # Range measurement noise std (m)
SIGMA_BEARING = 0.05        # Bearing measurement noise std (rad)
LANDMARKS = np.array([      # Landmark positions
    [10.0, 0.0],
    [0.0, 10.0],
    [10.0, 10.0],
])

# Sensor Fusion (Section 8)
DT_IMU = 0.01               # IMU rate: 100 Hz
DT_GPS_FUSION = 1.0         # GPS rate: 1 Hz
T_FUSION = 60.0             # Total simulation time (s)
SIGMA_IMU_ACC = 0.1         # IMU accelerometer noise std (m/s^2)
IMU_BIAS_DRIFT = 0.02       # IMU bias random walk (m/s^2/sqrt(s))
SIGMA_GPS_FUSION = 3.0      # GPS position noise std (m)

In [ ]:
# ---- Bayesian Fusion Visualization ----

def gaussian_pdf(x, mu, sigma2):
    """Evaluate Gaussian PDF.

    Args:
        x: Evaluation points. Shape: (N,).
        mu: Mean. Scalar.
        sigma2: Variance. Scalar.

    Returns:
        PDF values. Shape: (N,).
    """
    return (1.0 / np.sqrt(2 * np.pi * sigma2)) * np.exp(-0.5 * (x - mu)**2 / sigma2)


# Prior and measurement parameters
mu_prior = 5.0
sigma2_prior = 4.0  # sigma = 2
z_meas = 7.5
sigma2_meas = 1.0   # sigma = 1

# Posterior (precision-weighted fusion)
sigma2_post = 1.0 / (1.0 / sigma2_prior + 1.0 / sigma2_meas)
mu_post = sigma2_post * (mu_prior / sigma2_prior + z_meas / sigma2_meas)

x_range = np.linspace(0, 12, 500)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left panel: prior, likelihood, posterior
ax = axes[0]
ax.plot(x_range, gaussian_pdf(x_range, mu_prior, sigma2_prior),
        color='steelblue', linewidth=2, label=f'Prior: $\\mu$={mu_prior}, $\\sigma$={np.sqrt(sigma2_prior):.1f}')
ax.plot(x_range, gaussian_pdf(x_range, z_meas, sigma2_meas),
        color='coral', linewidth=2, label=f'Likelihood: z={z_meas}, $\\sigma$={np.sqrt(sigma2_meas):.1f}')
ax.plot(x_range, gaussian_pdf(x_range, mu_post, sigma2_post),
        color='seagreen', linewidth=2.5, label=f'Posterior: $\\mu$={mu_post:.2f}, $\\sigma$={np.sqrt(sigma2_post):.2f}')
ax.axvline(mu_prior, color='steelblue', linestyle=':', alpha=0.5)
ax.axvline(z_meas, color='coral', linestyle=':', alpha=0.5)
ax.axvline(mu_post, color='seagreen', linestyle=':', alpha=0.5)
ax.set_xlabel('State x')
ax.set_ylabel('Probability Density')
ax.set_title('Bayesian Fusion of Gaussians')
ax.legend()

# Right panel: show precision weighting
ax = axes[1]
precisions = [1.0 / sigma2_prior, 1.0 / sigma2_meas, 1.0 / sigma2_post]
labels = ['Prior', 'Measurement', 'Posterior']
colors = ['steelblue', 'coral', 'seagreen']
bars = ax.bar(labels, precisions, color=colors, alpha=0.8, edgecolor='black')
ax.set_ylabel('Precision ($1/\\sigma^2$)')
ax.set_title('Precision Addition: $\\tau_{post} = \\tau_{prior} + \\tau_{meas}$')
for bar, p in zip(bars, precisions):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{p:.2f}', ha='center', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"Prior:     mu = {mu_prior:.2f}, sigma^2 = {sigma2_prior:.2f}")
print(f"Measurement: z = {z_meas:.2f}, sigma^2 = {sigma2_meas:.2f}")
print(f"Posterior:  mu = {mu_post:.2f}, sigma^2 = {sigma2_post:.2f}")
print(f"Posterior is closer to measurement (higher precision source) [{('PASS' if abs(mu_post - z_meas) < abs(mu_post - mu_prior) else 'FAIL')}]")

---
## 3. The Kalman Filter Equations

### State-Space Model

A discrete-time linear dynamical system is described by:

$$x_{k+1} = F \, x_k + B \, u_k + w_k \qquad \text{(state transition)}$$

$$z_k = H \, x_k + v_k \qquad \text{(measurement model)}$$

where:

| Symbol | Dimension | Meaning |
|--------|-----------|--------|
| $x_k$ | $n \times 1$ | State vector |
| $u_k$ | $m \times 1$ | Control input |
| $z_k$ | $p \times 1$ | Measurement vector |
| $F$ | $n \times n$ | State transition matrix |
| $B$ | $n \times m$ | Control input matrix |
| $H$ | $p \times n$ | Measurement matrix |
| $w_k$ | $n \times 1$ | Process noise, $w_k \sim \mathcal{N}(0, Q)$ |
| $v_k$ | $p \times 1$ | Measurement noise, $v_k \sim \mathcal{N}(0, R)$ |

### Predict Step (Time Update)

Propagate the state estimate and covariance forward through the dynamics:

$$\boxed{\hat{x}_{k|k-1} = F \, \hat{x}_{k-1|k-1} + B \, u_k}$$

$$\boxed{P_{k|k-1} = F \, P_{k-1|k-1} \, F^T + Q}$$

### Update Step (Measurement Update)

Fuse the prediction with the new measurement:

$$y_k = z_k - H \, \hat{x}_{k|k-1} \qquad \text{(innovation / measurement residual)}$$

$$S_k = H \, P_{k|k-1} \, H^T + R \qquad \text{(innovation covariance)}$$

$$\boxed{K_k = P_{k|k-1} \, H^T \, S_k^{-1} \qquad \text{(Kalman gain)}}$$

$$\boxed{\hat{x}_{k|k} = \hat{x}_{k|k-1} + K_k \, y_k}$$

$$\boxed{P_{k|k} = (I - K_k \, H) \, P_{k|k-1}}$$

### MMSE Derivation Sketch

The Kalman gain $K_k$ is derived by minimizing the **minimum mean-squared error** (MMSE):

$$K_k = \arg\min_K \, \mathbb{E}\left[\| x_k - \hat{x}_{k|k} \|^2\right]$$

Substituting $\hat{x}_{k|k} = \hat{x}_{k|k-1} + K(z_k - H \hat{x}_{k|k-1})$ and differentiating the trace of the posterior covariance with respect to $K$, setting to zero yields the Kalman gain formula above. The key identity used is:

$$\frac{\partial}{\partial K} \text{tr}(P_{k|k}) = -2 P_{k|k-1} H^T + 2 K S_k = 0 \implies K_k = P_{k|k-1} H^T S_k^{-1}$$

---
## 4. Implementation from Scratch

In [ ]:
class KalmanFilter:
    """Linear Kalman Filter implementation.

    Implements the standard predict-update cycle for linear-Gaussian
    state-space models.

    Args:
        F: State transition matrix. Shape: (n, n).
        H: Measurement matrix. Shape: (p, n).
        Q: Process noise covariance. Shape: (n, n).
        R: Measurement noise covariance. Shape: (p, p).
        B: Control input matrix. Shape: (n, m). Optional.
        x0: Initial state estimate. Shape: (n,). Optional.
        P0: Initial covariance estimate. Shape: (n, n). Optional.
    """

    def __init__(self, F, H, Q, R, B=None, x0=None, P0=None):
        self.F = np.array(F, dtype=float)
        self.H = np.array(H, dtype=float)
        self.Q = np.array(Q, dtype=float)
        self.R = np.array(R, dtype=float)

        self.n = self.F.shape[0]  # state dimension
        self.p = self.H.shape[0]  # measurement dimension

        self.B = np.array(B, dtype=float) if B is not None else np.zeros((self.n, 1))
        self.x = np.array(x0, dtype=float) if x0 is not None else np.zeros(self.n)
        self.P = np.array(P0, dtype=float) if P0 is not None else np.eye(self.n) * 1000.0

    def predict(self, u=None):
        """Predict step: propagate state and covariance through dynamics.

        Args:
            u: Control input. Shape: (m,). Optional.

        Returns:
            x_pred: Predicted state. Shape: (n,).
            P_pred: Predicted covariance. Shape: (n, n).
        """
        self.x = self.F @ self.x
        if u is not None:
            self.x += self.B @ np.atleast_1d(u)

        self.P = self.F @ self.P @ self.F.T + self.Q
        return self.x.copy(), self.P.copy()

    def update(self, z):
        """Update step: fuse prediction with measurement.

        Uses np.linalg.solve for numerical stability instead of
        explicit matrix inverse.

        Args:
            z: Measurement vector. Shape: (p,).

        Returns:
            x_upd: Updated state. Shape: (n,).
            P_upd: Updated covariance. Shape: (n, n).
            innovation: Measurement residual. Shape: (p,).
            S: Innovation covariance. Shape: (p, p).
        """
        z = np.atleast_1d(z)

        # Innovation
        innovation = z - self.H @ self.x

        # Innovation covariance
        S = self.H @ self.P @ self.H.T + self.R

        # Kalman gain: K = P @ H^T @ S^{-1}
        # Solve S^T @ K^T = H @ P^T for K^T (more numerically stable)
        K = np.linalg.solve(S.T, (self.P @ self.H.T).T).T

        # State update
        self.x = self.x + K @ innovation

        # Covariance update (Joseph form for numerical stability)
        I_KH = np.eye(self.n) - K @ self.H
        self.P = I_KH @ self.P

        return self.x.copy(), self.P.copy(), innovation, S

    def filter(self, measurements, controls=None):
        """Run the full Kalman filter on a sequence of measurements.

        Args:
            measurements: Sequence of measurements. Shape: (T, p).
            controls: Sequence of control inputs. Shape: (T, m). Optional.

        Returns:
            states: State estimates at each step. Shape: (T, n).
            covariances: Covariance matrices at each step. Shape: (T, n, n).
            innovations: Innovation vectors at each step. Shape: (T, p).
            innovation_covs: Innovation covariances at each step. Shape: (T, p, p).
        """
        T = len(measurements)
        states = np.zeros((T, self.n))
        covariances = np.zeros((T, self.n, self.n))
        innovations = np.zeros((T, self.p))
        innovation_covs = np.zeros((T, self.p, self.p))

        for k in range(T):
            u = controls[k] if controls is not None else None
            self.predict(u)
            x_upd, P_upd, innov, S = self.update(measurements[k])

            states[k] = x_upd
            covariances[k] = P_upd
            innovations[k] = innov
            innovation_covs[k] = S

        return states, covariances, innovations, innovation_covs

In [ ]:
# ---- Application: 1D Falling Object ----

# State: [position, velocity]
# x_{k+1} = F*x_k + B*u_k + w_k
# z_k = H*x_k + v_k

dt = DT_FALL
N_steps = int(T_FALL / dt)
time_fall = np.arange(N_steps) * dt

# System matrices
F_fall = np.array([[1.0, dt],
                   [0.0, 1.0]])

B_fall = np.array([[0.5 * dt**2],
                   [dt]])

H_fall = np.array([[1.0, 0.0]])  # measure position only

Q_fall = np.array([[Q_FALL_POS, 0.0],
                   [0.0, Q_FALL_VEL]])

R_fall = np.array([[R_FALL]])

u_fall = np.array([-GRAVITY])  # control input: gravitational acceleration (downward)

# ---- Generate True Trajectory ----
true_states = np.zeros((N_steps, 2))
true_states[0] = [X0_FALL, V0_FALL]

for k in range(1, N_steps):
    true_states[k] = F_fall @ true_states[k-1] + B_fall.flatten() * (-GRAVITY)
    true_states[k] += np.random.multivariate_normal(np.zeros(2), Q_fall)

# ---- Generate Noisy Measurements ----
measurements_fall = true_states[:, 0] + np.random.normal(0, np.sqrt(R_FALL), N_steps)

# ---- Run Kalman Filter ----
kf = KalmanFilter(
    F=F_fall, H=H_fall, Q=Q_fall, R=R_fall, B=B_fall,
    x0=np.array([X0_FALL, V0_FALL]),
    P0=np.diag([10.0, 1.0])
)

controls_fall = np.full((N_steps, 1), -GRAVITY)
est_states, est_covs, innovations_fall, innov_covs_fall = kf.filter(
    measurements_fall.reshape(-1, 1), controls_fall
)

# ---- RMSE Verification ----
rmse_pos = np.sqrt(np.mean((est_states[:, 0] - true_states[:, 0])**2))
rmse_vel = np.sqrt(np.mean((est_states[:, 1] - true_states[:, 1])**2))
rmse_meas = np.sqrt(np.mean((measurements_fall - true_states[:, 0])**2))

print(f"Position RMSE (KF):          {rmse_pos:.4f} m")
print(f"Position RMSE (raw meas):    {rmse_meas:.4f} m")
print(f"Velocity RMSE (KF):          {rmse_vel:.4f} m/s")
status_kf = "PASS" if rmse_pos < rmse_meas else "FAIL"
print(f"KF improves over raw measurements: {rmse_pos:.4f} < {rmse_meas:.4f} [{status_kf}]")

In [ ]:
# ---- 3-Panel Figure: Falling Object Tracking ----
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: Position tracking
ax = axes[0]
ax.plot(time_fall, true_states[:, 0], color='steelblue', linewidth=2, label='True position')
ax.scatter(time_fall, measurements_fall, color='coral', s=8, alpha=0.5, label='Measurements', zorder=3)
ax.plot(time_fall, est_states[:, 0], color='seagreen', linewidth=2, label='KF estimate')
# 2-sigma bounds
sigma_pos = np.sqrt(est_covs[:, 0, 0])
ax.fill_between(time_fall, est_states[:, 0] - 2*sigma_pos, est_states[:, 0] + 2*sigma_pos,
                color='seagreen', alpha=0.15, label='$\\pm 2\\sigma$')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Position (m)')
ax.set_title('Position Tracking')
ax.legend(fontsize=9)

# Panel 2: Velocity estimation
ax = axes[1]
ax.plot(time_fall, true_states[:, 1], color='steelblue', linewidth=2, label='True velocity')
ax.plot(time_fall, est_states[:, 1], color='seagreen', linewidth=2, label='KF estimate')
sigma_vel = np.sqrt(est_covs[:, 1, 1])
ax.fill_between(time_fall, est_states[:, 1] - 2*sigma_vel, est_states[:, 1] + 2*sigma_vel,
                color='seagreen', alpha=0.15, label='$\\pm 2\\sigma$')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Velocity (m/s)')
ax.set_title('Velocity Estimation (unobserved)')
ax.legend(fontsize=9)

# Panel 3: Kalman gain evolution
ax = axes[2]
# Re-run to get Kalman gain history
kf_trace = KalmanFilter(F=F_fall, H=H_fall, Q=Q_fall, R=R_fall, B=B_fall,
                        x0=np.array([X0_FALL, V0_FALL]), P0=np.diag([10.0, 1.0]))
K_history = []
for k in range(N_steps):
    kf_trace.predict(controls_fall[k])
    # Compute Kalman gain before update
    S = kf_trace.H @ kf_trace.P @ kf_trace.H.T + kf_trace.R
    K = np.linalg.solve(S.T, (kf_trace.P @ kf_trace.H.T).T).T
    K_history.append(K.flatten())
    kf_trace.update(measurements_fall[k])

K_history = np.array(K_history)
ax.plot(time_fall, K_history[:, 0], color='steelblue', linewidth=2, label='$K_{pos}$')
ax.plot(time_fall, K_history[:, 1], color='coral', linewidth=2, label='$K_{vel}$')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Kalman Gain')
ax.set_title('Kalman Gain Convergence')
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

---
## 5. Innovation Sequence Analysis

How do we know if our Kalman Filter is working correctly? The **innovation sequence** (measurement residuals $y_k$) provides a built-in consistency check.

### Normalized Innovation Squared (NIS)

For a correctly-tuned Kalman Filter, the innovations should be **zero-mean** and **white** (uncorrelated in time), with covariance $S_k$. The **Normalized Innovation Squared** (NIS) is:

$$\varepsilon_k = y_k^T \, S_k^{-1} \, y_k$$

If the filter is consistent, then:

$$\boxed{\varepsilon_k \sim \chi^2(n_z)}$$

where $n_z$ is the measurement dimension. This means approximately 95% of NIS values should fall below the $\chi^2_{n_z}(0.95)$ threshold.

This serves as the "gradient check" analog for filtering — a quick numerical test that catches implementation bugs.

In [ ]:
def compute_nis(innovations, innovation_covs):
    """Compute Normalized Innovation Squared (NIS) for filter consistency.

    Args:
        innovations: Innovation vectors. Shape: (T, p).
        innovation_covs: Innovation covariances. Shape: (T, p, p).

    Returns:
        nis: NIS values. Shape: (T,).
    """
    T = len(innovations)
    nis = np.zeros(T)
    for k in range(T):
        y = innovations[k]
        S = innovation_covs[k]
        # NIS = y^T S^{-1} y, using solve for stability
        nis[k] = y @ np.linalg.solve(S, y)
    return nis


# ---- NIS for Falling Object ----
nis_fall = compute_nis(innovations_fall, innov_covs_fall)
nz_fall = 1  # measurement dimension
chi2_bound_95 = chi2.ppf(0.95, nz_fall)
fraction_within = np.mean(nis_fall < chi2_bound_95)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: NIS over time
ax = axes[0]
ax.plot(time_fall, nis_fall, color='steelblue', linewidth=1, alpha=0.7, label='NIS')
ax.axhline(chi2_bound_95, color='coral', linestyle='--', linewidth=2,
           label=f'95% $\\chi^2$ bound = {chi2_bound_95:.2f}')
ax.set_xlabel('Time (s)')
ax.set_ylabel('NIS ($\\varepsilon_k$)')
ax.set_title('Normalized Innovation Squared')
ax.legend()

# Right: Innovation histogram
ax = axes[1]
innov_normalized = innovations_fall[:, 0] / np.sqrt(innov_covs_fall[:, 0, 0])
ax.hist(innov_normalized, bins=30, density=True, color='steelblue', alpha=0.7, edgecolor='black', label='Normalized innovations')
x_norm = np.linspace(-4, 4, 200)
ax.plot(x_norm, norm.pdf(x_norm), color='coral', linewidth=2, label='$\\mathcal{N}(0,1)$')
ax.set_xlabel('Normalized Innovation')
ax.set_ylabel('Density')
ax.set_title('Innovation Distribution')
ax.legend()

plt.tight_layout()
plt.show()

status_nis = "PASS" if 0.85 < fraction_within < 1.0 else "FAIL"
print(f"NIS consistency test: {fraction_within:.1%} of NIS values within 95% bound [{status_nis}]")

---
## 6. Application: 2D Robot Localization

Now we tackle a more realistic scenario: a robot moving in 2D, tracked by **two different sensors** with different noise characteristics and update rates.

### System Model

- **State:** $x = [p_x, p_y, v_x, v_y]^T$ (position and velocity in 2D)
- **Constant-velocity model:** $F = \begin{bmatrix} I_2 & \Delta t \cdot I_2 \\ 0 & I_2 \end{bmatrix}$
- **Sensor 1 (GPS):** Measures position $[p_x, p_y]$, noisy ($\sigma = 5$ m), low rate (every 10 steps)
- **Sensor 2 (Odometry):** Measures velocity $[v_x, v_y]$, less noisy ($\sigma = 0.5$ m/s), every step

The key challenge: GPS is accurate over long time scales but slow and noisy; odometry is fast and smooth but drifts. The Kalman Filter optimally combines both.

In [ ]:
# ---- 2D Robot Localization ----

N_loc = int(T_LOC / DT_LOC)
time_loc = np.arange(N_loc) * DT_LOC

# ---- Generate True Circular Trajectory ----
omega_path = 2 * np.pi / PERIOD_PATH  # angular velocity
true_px = RADIUS_PATH * np.cos(omega_path * time_loc)
true_py = RADIUS_PATH * np.sin(omega_path * time_loc)
true_vx = -RADIUS_PATH * omega_path * np.sin(omega_path * time_loc)
true_vy = RADIUS_PATH * omega_path * np.cos(omega_path * time_loc)
true_states_2d = np.column_stack([true_px, true_py, true_vx, true_vy])

# ---- System Matrices ----
F_2d = np.array([
    [1, 0, DT_LOC, 0],
    [0, 1, 0, DT_LOC],
    [0, 0, 1, 0],
    [0, 0, 0, 1]
])

# Process noise (accounts for constant-velocity model mismatch for circular motion)
q_pos = 0.1
q_vel = 1.0
Q_2d = np.diag([q_pos, q_pos, q_vel, q_vel])

# GPS sensor: measures position
H_gps = np.array([
    [1, 0, 0, 0],
    [0, 1, 0, 0]
])
R_gps = np.diag([SIGMA_GPS**2, SIGMA_GPS**2])

# Odometry sensor: measures velocity
H_odom = np.array([
    [0, 0, 1, 0],
    [0, 0, 0, 1]
])
R_odom = np.diag([SIGMA_ODOM**2, SIGMA_ODOM**2])

# ---- Generate Measurements ----
gps_measurements = true_states_2d[:, :2] + np.random.normal(0, SIGMA_GPS, (N_loc, 2))
odom_measurements = true_states_2d[:, 2:4] + np.random.normal(0, SIGMA_ODOM, (N_loc, 2))

# ---- Run KF with Multi-Sensor Updates ----
x0_2d = np.array([RADIUS_PATH, 0.0, 0.0, RADIUS_PATH * omega_path])
P0_2d = np.diag([10.0, 10.0, 1.0, 1.0])

# Manual filter loop for multi-sensor updates
kf_2d = KalmanFilter(F=F_2d, H=H_gps, Q=Q_2d, R=R_gps, x0=x0_2d, P0=P0_2d)

est_states_2d = np.zeros((N_loc, 4))
est_covs_2d = np.zeros((N_loc, 4, 4))
innovations_2d = []
innov_covs_2d = []

# Dead reckoning (odometry-only) for comparison
dead_reckoning = np.zeros((N_loc, 4))
dead_reckoning[0] = x0_2d

for k in range(N_loc):
    # Predict
    kf_2d.predict()

    # Always update with odometry
    kf_2d.H = H_odom
    kf_2d.R = R_odom
    kf_2d.update(odom_measurements[k])

    # Update with GPS every GPS_INTERVAL steps
    if k % GPS_INTERVAL == 0:
        kf_2d.H = H_gps
        kf_2d.R = R_gps
        x_upd, P_upd, innov, S = kf_2d.update(gps_measurements[k])
        innovations_2d.append(innov)
        innov_covs_2d.append(S)

    est_states_2d[k] = kf_2d.x.copy()
    est_covs_2d[k] = kf_2d.P.copy()

    # Dead reckoning: integrate odometry
    if k > 0:
        dead_reckoning[k, 2:4] = odom_measurements[k]  # use measured velocity
        dead_reckoning[k, 0] = dead_reckoning[k-1, 0] + odom_measurements[k, 0] * DT_LOC
        dead_reckoning[k, 1] = dead_reckoning[k-1, 1] + odom_measurements[k, 1] * DT_LOC

# ---- Compute RMSE ----
rmse_kf_2d = np.sqrt(np.mean((est_states_2d[:, :2] - true_states_2d[:, :2])**2))
rmse_gps = np.sqrt(np.mean((gps_measurements - true_states_2d[:, :2])**2))
rmse_dr = np.sqrt(np.mean((dead_reckoning[:, :2] - true_states_2d[:, :2])**2))

print(f"Position RMSE (KF fused):     {rmse_kf_2d:.4f} m")
print(f"Position RMSE (GPS only):     {rmse_gps:.4f} m")
print(f"Position RMSE (dead reckoning): {rmse_dr:.4f} m")
status_2d = "PASS" if rmse_kf_2d < rmse_gps and rmse_kf_2d < rmse_dr else "FAIL"
print(f"KF outperforms both individual sensors: [{status_2d}]")

In [ ]:
# ---- Multi-Panel: 2D Localization Results ----
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Panel 1: Trajectory plot
ax = axes[0]
ax.plot(true_states_2d[:, 0], true_states_2d[:, 1], color='steelblue', linewidth=2, label='True path')
gps_idx = np.arange(0, N_loc, GPS_INTERVAL)
ax.scatter(gps_measurements[gps_idx, 0], gps_measurements[gps_idx, 1],
           color='coral', s=30, alpha=0.6, marker='^', label='GPS measurements', zorder=3)
ax.plot(dead_reckoning[:, 0], dead_reckoning[:, 1], color='goldenrod',
        linewidth=1.5, linestyle=':', label='Dead reckoning')
ax.plot(est_states_2d[:, 0], est_states_2d[:, 1], color='seagreen',
        linewidth=2, label='KF estimate')
ax.set_xlabel('x (m)')
ax.set_ylabel('y (m)')
ax.set_title('2D Robot Localization')
ax.legend(fontsize=9)
ax.set_aspect('equal')

# Panel 2: x-position error with 2-sigma bounds
ax = axes[1]
err_x = est_states_2d[:, 0] - true_states_2d[:, 0]
sigma_x = np.sqrt(est_covs_2d[:, 0, 0])
ax.plot(time_loc, err_x, color='steelblue', linewidth=1.5, label='x error')
ax.fill_between(time_loc, -2*sigma_x, 2*sigma_x, color='coral', alpha=0.2, label='$\\pm 2\\sigma$')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Error (m)')
ax.set_title('X-Position Error')
ax.legend()

# Panel 3: y-position error with 2-sigma bounds
ax = axes[2]
err_y = est_states_2d[:, 1] - true_states_2d[:, 1]
sigma_y = np.sqrt(est_covs_2d[:, 1, 1])
ax.plot(time_loc, err_y, color='steelblue', linewidth=1.5, label='y error')
ax.fill_between(time_loc, -2*sigma_y, 2*sigma_y, color='coral', alpha=0.2, label='$\\pm 2\\sigma$')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Error (m)')
ax.set_title('Y-Position Error')
ax.legend()

plt.tight_layout()
plt.show()

---
## 7. Extended Kalman Filter (EKF)

Many robotic systems have **nonlinear** dynamics or measurements. The EKF handles this by **linearizing** around the current estimate.

### Nonlinear State-Space Model

$$x_{k+1} = f(x_k, u_k) + w_k \qquad z_k = h(x_k) + v_k$$

### EKF Algorithm

**Predict:**
$$\hat{x}_{k|k-1} = f(\hat{x}_{k-1|k-1}, u_k)$$
$$P_{k|k-1} = F_k \, P_{k-1|k-1} \, F_k^T + Q$$

where $F_k = \frac{\partial f}{\partial x}\bigg|_{\hat{x}_{k-1|k-1}, u_k}$ is the Jacobian of the dynamics.

**Update:**
$$y_k = z_k - h(\hat{x}_{k|k-1})$$
$$S_k = H_k \, P_{k|k-1} \, H_k^T + R$$
$$K_k = P_{k|k-1} \, H_k^T \, S_k^{-1}$$
$$\hat{x}_{k|k} = \hat{x}_{k|k-1} + K_k \, y_k$$
$$P_{k|k} = (I - K_k H_k) \, P_{k|k-1}$$

where $H_k = \frac{\partial h}{\partial x}\bigg|_{\hat{x}_{k|k-1}}$ is the Jacobian of the measurement model.

### Application: Unicycle Model Robot

**State:** $x = [p_x, p_y, \theta]^T$ (position and heading)

**Control:** $u = [v, \omega]^T$ (linear and angular velocity)

**Dynamics:**
$$f(x, u) = \begin{bmatrix} p_x + v \cos\theta \cdot \Delta t \\ p_y + v \sin\theta \cdot \Delta t \\ \theta + \omega \cdot \Delta t \end{bmatrix}$$

**Measurement:** Range-bearing to $L$ known landmarks:
$$h_i(x) = \begin{bmatrix} \sqrt{(p_x - l_{x,i})^2 + (p_y - l_{y,i})^2} \\ \text{atan2}(l_{y,i} - p_y, \; l_{x,i} - p_x) - \theta \end{bmatrix}$$

**Dynamics Jacobian:**
$$F = \frac{\partial f}{\partial x} = \begin{bmatrix} 1 & 0 & -v \sin\theta \cdot \Delta t \\ 0 & 1 & v \cos\theta \cdot \Delta t \\ 0 & 0 & 1 \end{bmatrix}$$

**Measurement Jacobian** (for landmark $i$):
$$H_i = \begin{bmatrix} \frac{p_x - l_{x,i}}{r_i} & \frac{p_y - l_{y,i}}{r_i} & 0 \\ \frac{-(p_y - l_{y,i})}{r_i^2} & \frac{p_x - l_{x,i}}{r_i^2} & -1 \end{bmatrix}$$

where $r_i = \sqrt{(p_x - l_{x,i})^2 + (p_y - l_{y,i})^2}$.

In [ ]:
class ExtendedKalmanFilter:
    """Extended Kalman Filter for nonlinear systems.

    Uses first-order Taylor expansion (Jacobian linearization) to
    approximate the nonlinear system as locally linear.

    Args:
        f: Nonlinear state transition function. Signature: f(x, u) -> x_next.
        h: Nonlinear measurement function. Signature: h(x) -> z.
        F_jac: Jacobian of f w.r.t. x. Signature: F_jac(x, u) -> Shape: (n, n).
        H_jac: Jacobian of h w.r.t. x. Signature: H_jac(x) -> Shape: (p, n).
        Q: Process noise covariance. Shape: (n, n).
        R: Measurement noise covariance. Shape: (p, p).
        x0: Initial state estimate. Shape: (n,).
        P0: Initial covariance estimate. Shape: (n, n).
    """

    def __init__(self, f, h, F_jac, H_jac, Q, R, x0, P0):
        self.f = f
        self.h = h
        self.F_jac = F_jac
        self.H_jac = H_jac
        self.Q = np.array(Q, dtype=float)
        self.R = np.array(R, dtype=float)
        self.x = np.array(x0, dtype=float)
        self.P = np.array(P0, dtype=float)
        self.n = len(x0)

    def predict(self, u=None):
        """EKF predict step.

        Args:
            u: Control input. Shape: (m,). Optional.

        Returns:
            x_pred: Predicted state. Shape: (n,).
            P_pred: Predicted covariance. Shape: (n, n).
        """
        F = self.F_jac(self.x, u)
        self.x = self.f(self.x, u)
        self.P = F @ self.P @ F.T + self.Q
        return self.x.copy(), self.P.copy()

    def update(self, z):
        """EKF update step.

        Args:
            z: Measurement vector. Shape: (p,).

        Returns:
            x_upd: Updated state. Shape: (n,).
            P_upd: Updated covariance. Shape: (n, n).
            innovation: Measurement residual. Shape: (p,).
            S: Innovation covariance. Shape: (p, p).
        """
        z = np.atleast_1d(z)
        H = self.H_jac(self.x)

        # Innovation
        z_pred = self.h(self.x)
        innovation = z - z_pred

        # Wrap bearing angles to [-pi, pi]
        for i in range(len(innovation)):
            if i % 2 == 1:  # bearing components (odd indices for range-bearing)
                innovation[i] = (innovation[i] + np.pi) % (2 * np.pi) - np.pi

        # Innovation covariance
        S = H @ self.P @ H.T + self.R

        # Kalman gain
        K = np.linalg.solve(S.T, (self.P @ H.T).T).T

        # State update
        self.x = self.x + K @ innovation

        # Wrap theta to [-pi, pi]
        self.x[2] = (self.x[2] + np.pi) % (2 * np.pi) - np.pi

        # Covariance update
        I_KH = np.eye(self.n) - K @ H
        self.P = I_KH @ self.P

        return self.x.copy(), self.P.copy(), innovation, S

In [ ]:
# ---- Unicycle Model Functions ----

def unicycle_dynamics(x, u):
    """Unicycle model state transition.

    Args:
        x: State [px, py, theta]. Shape: (3,).
        u: Control [v, omega]. Shape: (2,).

    Returns:
        x_next: Next state. Shape: (3,).
    """
    px, py, theta = x
    v, omega = u
    return np.array([
        px + v * np.cos(theta) * DT_UNI,
        py + v * np.sin(theta) * DT_UNI,
        theta + omega * DT_UNI
    ])


def unicycle_dynamics_jacobian(x, u):
    """Jacobian of unicycle dynamics w.r.t. state.

    Args:
        x: State [px, py, theta]. Shape: (3,).
        u: Control [v, omega]. Shape: (2,).

    Returns:
        F: Jacobian matrix. Shape: (3, 3).
    """
    _, _, theta = x
    v, _ = u
    return np.array([
        [1, 0, -v * np.sin(theta) * DT_UNI],
        [0, 1,  v * np.cos(theta) * DT_UNI],
        [0, 0, 1]
    ])


def range_bearing_measurement(x):
    """Range-bearing measurement to all landmarks.

    Args:
        x: State [px, py, theta]. Shape: (3,).

    Returns:
        z: Measurements [r1, b1, r2, b2, ...]. Shape: (2*L,).
    """
    px, py, theta = x
    z = []
    for lm in LANDMARKS:
        dx = lm[0] - px
        dy = lm[1] - py
        r = np.sqrt(dx**2 + dy**2)
        b = np.arctan2(dy, dx) - theta
        z.extend([r, b])
    return np.array(z)


def range_bearing_jacobian(x):
    """Jacobian of range-bearing measurement w.r.t. state.

    Args:
        x: State [px, py, theta]. Shape: (3,).

    Returns:
        H: Jacobian matrix. Shape: (2*L, 3).
    """
    px, py, theta = x
    H = []
    for lm in LANDMARKS:
        dx = lm[0] - px
        dy = lm[1] - py
        r = np.sqrt(dx**2 + dy**2)
        # Range row: dr/dpx, dr/dpy, dr/dtheta
        H.append([-dx / r, -dy / r, 0])
        # Bearing row: db/dpx, db/dpy, db/dtheta
        H.append([dy / r**2, -dx / r**2, -1])
    return np.array(H)


# ---- Jacobian Verification via Finite Differences ----
x_test_uni = np.array([3.0, 4.0, 0.5])
u_test_uni = np.array([1.0, 0.3])

# Dynamics Jacobian
F_analytical = unicycle_dynamics_jacobian(x_test_uni, u_test_uni)
F_numerical = np.zeros((3, 3))
for i in range(3):
    x_plus = x_test_uni.copy()
    x_minus = x_test_uni.copy()
    x_plus[i] += FD_EPSILON
    x_minus[i] -= FD_EPSILON
    F_numerical[:, i] = (unicycle_dynamics(x_plus, u_test_uni) -
                         unicycle_dynamics(x_minus, u_test_uni)) / (2 * FD_EPSILON)

err_F = np.max(np.abs(F_analytical - F_numerical))
status_F = "PASS" if err_F < 1e-5 else "FAIL"
print(f"Dynamics Jacobian verification: max error = {err_F:.2e} [{status_F}]")

# Measurement Jacobian
H_analytical = range_bearing_jacobian(x_test_uni)
H_numerical = np.zeros_like(H_analytical)
for i in range(3):
    x_plus = x_test_uni.copy()
    x_minus = x_test_uni.copy()
    x_plus[i] += FD_EPSILON
    x_minus[i] -= FD_EPSILON
    H_numerical[:, i] = (range_bearing_measurement(x_plus) -
                         range_bearing_measurement(x_minus)) / (2 * FD_EPSILON)

err_H = np.max(np.abs(H_analytical - H_numerical))
status_H = "PASS" if err_H < 1e-5 else "FAIL"
print(f"Measurement Jacobian verification: max error = {err_H:.2e} [{status_H}]")

In [ ]:
# ---- Simulate Unicycle Robot with EKF ----

N_uni = int(T_UNI / DT_UNI)
time_uni = np.arange(N_uni) * DT_UNI

# Control inputs: move in a figure-8 pattern
v_ctrl = 2.0 * np.ones(N_uni)  # constant forward velocity
omega_ctrl = 0.5 * np.sin(2 * np.pi * time_uni / T_UNI)  # varying angular velocity

# Process noise
Q_uni = np.diag([0.01, 0.01, 0.005])

# Measurement noise (per landmark: range, bearing)
n_landmarks = len(LANDMARKS)
R_uni = np.diag([SIGMA_RANGE**2, SIGMA_BEARING**2] * n_landmarks)

# ---- Generate True Trajectory ----
true_states_uni = np.zeros((N_uni, 3))
true_states_uni[0] = [0.0, 0.0, 0.0]

for k in range(1, N_uni):
    u_k = np.array([v_ctrl[k-1], omega_ctrl[k-1]])
    true_states_uni[k] = unicycle_dynamics(true_states_uni[k-1], u_k)
    true_states_uni[k] += np.random.multivariate_normal(np.zeros(3), Q_uni)

# ---- Generate Measurements ----
measurements_uni = np.zeros((N_uni, 2 * n_landmarks))
for k in range(N_uni):
    z_true = range_bearing_measurement(true_states_uni[k])
    noise = np.zeros(2 * n_landmarks)
    for i in range(n_landmarks):
        noise[2*i] = np.random.normal(0, SIGMA_RANGE)
        noise[2*i+1] = np.random.normal(0, SIGMA_BEARING)
    measurements_uni[k] = z_true + noise

# ---- Run EKF ----
ekf = ExtendedKalmanFilter(
    f=unicycle_dynamics,
    h=range_bearing_measurement,
    F_jac=unicycle_dynamics_jacobian,
    H_jac=range_bearing_jacobian,
    Q=Q_uni,
    R=R_uni,
    x0=np.array([0.5, 0.5, 0.1]),  # slightly wrong initial guess
    P0=np.diag([1.0, 1.0, 0.1])
)

est_states_uni = np.zeros((N_uni, 3))
est_covs_uni = np.zeros((N_uni, 3, 3))

for k in range(N_uni):
    u_k = np.array([v_ctrl[k], omega_ctrl[k]])
    ekf.predict(u_k)
    x_upd, P_upd, _, _ = ekf.update(measurements_uni[k])
    est_states_uni[k] = x_upd
    est_covs_uni[k] = P_upd

# ---- RMSE ----
rmse_ekf_pos = np.sqrt(np.mean((est_states_uni[:, :2] - true_states_uni[:, :2])**2))
rmse_ekf_theta = np.sqrt(np.mean(
    ((est_states_uni[:, 2] - true_states_uni[:, 2] + np.pi) % (2*np.pi) - np.pi)**2
))

print(f"EKF position RMSE:  {rmse_ekf_pos:.4f} m")
print(f"EKF heading RMSE:   {rmse_ekf_theta:.4f} rad ({np.degrees(rmse_ekf_theta):.2f} deg)")
status_ekf = "PASS" if rmse_ekf_pos < 1.0 else "FAIL"
print(f"EKF position RMSE < 1.0 m: [{status_ekf}]")

In [ ]:
def plot_uncertainty_ellipse(ax, mean, cov, n_std=2, **kwargs):
    """Plot an uncertainty ellipse.

    Args:
        ax: Matplotlib axis.
        mean: Center of ellipse. Shape: (2,).
        cov: 2x2 covariance matrix. Shape: (2, 2).
        n_std: Number of standard deviations for ellipse size. Scalar.
        **kwargs: Passed to matplotlib Ellipse.
    """
    eigvals, eigvecs = np.linalg.eigh(cov)
    # Eigenvalues are sorted ascending; use the larger for width
    order = eigvals.argsort()[::-1]
    eigvals = eigvals[order]
    eigvecs = eigvecs[:, order]

    angle = np.degrees(np.arctan2(eigvecs[1, 0], eigvecs[0, 0]))
    width = 2 * n_std * np.sqrt(eigvals[0])
    height = 2 * n_std * np.sqrt(eigvals[1])

    ellipse = Ellipse(xy=mean, width=width, height=height, angle=angle, **kwargs)
    ax.add_patch(ellipse)


# ---- EKF Unicycle Visualization ----
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: Trajectory with uncertainty ellipses
ax = axes[0]
ax.plot(true_states_uni[:, 0], true_states_uni[:, 1], color='steelblue',
        linewidth=2, label='True path')
ax.plot(est_states_uni[:, 0], est_states_uni[:, 1], color='seagreen',
        linewidth=2, label='EKF estimate')

# Plot uncertainty ellipses every 20 steps
for k in range(0, N_uni, 20):
    plot_uncertainty_ellipse(
        ax, est_states_uni[k, :2], est_covs_uni[k, :2, :2],
        n_std=2, fill=False, edgecolor='seagreen', alpha=0.4, linewidth=1
    )

# Plot landmarks
ax.scatter(LANDMARKS[:, 0], LANDMARKS[:, 1], color='coral', s=100,
           marker='*', zorder=5, label='Landmarks')
ax.set_xlabel('x (m)')
ax.set_ylabel('y (m)')
ax.set_title('EKF Unicycle Localization')
ax.legend(fontsize=9)
ax.set_aspect('equal')

# Right: Heading estimation
ax = axes[1]
ax.plot(time_uni, np.degrees(true_states_uni[:, 2]), color='steelblue',
        linewidth=2, label='True $\\theta$')
ax.plot(time_uni, np.degrees(est_states_uni[:, 2]), color='seagreen',
        linewidth=2, label='EKF estimate')
sigma_theta = np.sqrt(est_covs_uni[:, 2, 2])
ax.fill_between(time_uni,
                np.degrees(est_states_uni[:, 2] - 2*sigma_theta),
                np.degrees(est_states_uni[:, 2] + 2*sigma_theta),
                color='seagreen', alpha=0.15, label='$\\pm 2\\sigma$')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Heading (deg)')
ax.set_title('Heading Estimation')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

---
## 8. Sensor Fusion: IMU + GPS

Perhaps the most compelling application of the Kalman Filter is **sensor fusion** — combining multiple sensors with complementary strengths.

| Sensor | Strength | Weakness |
|--------|----------|----------|
| IMU (accelerometer) | High rate (100 Hz), smooth | Drift from bias and integration |
| GPS | No drift, absolute position | Low rate (1 Hz), noisy ($\sigma \sim 3$ m) |

### 1D Simplified Model

We use a 1D model to clearly demonstrate the fusion concept:
- **State:** $x = [p, v, a_{\text{bias}}]^T$ (position, velocity, accelerometer bias)
- **IMU** provides acceleration measurements at 100 Hz: $z_{\text{imu}} = a_{\text{true}} + a_{\text{bias}} + \text{noise}$
- **GPS** provides position measurements at 1 Hz: $z_{\text{gps}} = p + \text{noise}$

The bias is modeled as a **random walk**, which the filter can track and correct for.

In [ ]:
# ---- Sensor Fusion: IMU + GPS ----

N_fusion = int(T_FUSION / DT_IMU)
time_fusion = np.arange(N_fusion) * DT_IMU
gps_steps = int(DT_GPS_FUSION / DT_IMU)  # GPS updates every this many IMU steps

# ---- Generate True Trajectory ----
# Sinusoidal acceleration profile (interesting but smooth)
true_acc = 2.0 * np.sin(2 * np.pi * time_fusion / 15.0)  # period = 15s

true_pos = np.zeros(N_fusion)
true_vel = np.zeros(N_fusion)
for k in range(1, N_fusion):
    true_vel[k] = true_vel[k-1] + true_acc[k-1] * DT_IMU
    true_pos[k] = true_pos[k-1] + true_vel[k-1] * DT_IMU + 0.5 * true_acc[k-1] * DT_IMU**2

# ---- Generate IMU Bias (random walk) ----
imu_bias = np.zeros(N_fusion)
for k in range(1, N_fusion):
    imu_bias[k] = imu_bias[k-1] + IMU_BIAS_DRIFT * np.sqrt(DT_IMU) * np.random.randn()

# ---- Generate Sensor Measurements ----
imu_measurements = true_acc + imu_bias + np.random.normal(0, SIGMA_IMU_ACC, N_fusion)
gps_measurements_fusion = true_pos + np.random.normal(0, SIGMA_GPS_FUSION, N_fusion)

# ---- State: [position, velocity, accel_bias] ----
dt = DT_IMU

# System matrices for IMU propagation
F_fusion = np.array([
    [1, dt, -0.5*dt**2],
    [0, 1,  -dt],
    [0, 0,  1]
])

B_fusion = np.array([
    [0.5*dt**2],
    [dt],
    [0]
])

# Process noise
Q_fusion = np.diag([
    (0.5*dt**2 * SIGMA_IMU_ACC)**2,  # position process noise
    (dt * SIGMA_IMU_ACC)**2,           # velocity process noise
    (IMU_BIAS_DRIFT * np.sqrt(dt))**2  # bias random walk
])

# GPS measurement matrix
H_gps_fusion = np.array([[1, 0, 0]])
R_gps_fusion = np.array([[SIGMA_GPS_FUSION**2]])

# ---- Run Fusion EKF ----
x_fused = np.zeros(3)  # start at origin
P_fused = np.diag([1.0, 0.1, 0.01])

est_fused = np.zeros((N_fusion, 3))
est_P_fused = np.zeros((N_fusion, 3, 3))

for k in range(N_fusion):
    # Predict with IMU measurement as control input
    x_fused = F_fusion @ x_fused + B_fusion.flatten() * imu_measurements[k]
    P_fused = F_fusion @ P_fused @ F_fusion.T + Q_fusion

    # GPS update (every gps_steps)
    if k % gps_steps == 0:
        z_gps = gps_measurements_fusion[k]
        S = H_gps_fusion @ P_fused @ H_gps_fusion.T + R_gps_fusion
        K = np.linalg.solve(S.T, (P_fused @ H_gps_fusion.T).T).T
        innovation = z_gps - H_gps_fusion @ x_fused
        x_fused = x_fused + K.flatten() * innovation.item()
        P_fused = (np.eye(3) - K @ H_gps_fusion) @ P_fused

    est_fused[k] = x_fused
    est_P_fused[k] = P_fused

# ---- IMU-only integration (for comparison) ----
imu_only_pos = np.zeros(N_fusion)
imu_only_vel = np.zeros(N_fusion)
for k in range(1, N_fusion):
    imu_only_vel[k] = imu_only_vel[k-1] + imu_measurements[k-1] * DT_IMU
    imu_only_pos[k] = imu_only_pos[k-1] + imu_only_vel[k-1] * DT_IMU

# ---- GPS-only (sample-and-hold) ----
gps_only_pos = np.zeros(N_fusion)
for k in range(N_fusion):
    if k % gps_steps == 0:
        gps_only_pos[k] = gps_measurements_fusion[k]
    else:
        gps_only_pos[k] = gps_only_pos[k - 1] if k > 0 else 0

# ---- RMSE Comparison ----
rmse_fused = np.sqrt(np.mean((est_fused[:, 0] - true_pos)**2))
rmse_imu_only = np.sqrt(np.mean((imu_only_pos - true_pos)**2))
rmse_gps_only = np.sqrt(np.mean((gps_only_pos - true_pos)**2))

print(f"Position RMSE (fused):    {rmse_fused:.4f} m")
print(f"Position RMSE (IMU only): {rmse_imu_only:.4f} m")
print(f"Position RMSE (GPS only): {rmse_gps_only:.4f} m")
status_fusion = "PASS" if rmse_fused < rmse_imu_only and rmse_fused < rmse_gps_only else "FAIL"
print(f"Fusion outperforms individual sensors: [{status_fusion}]")

In [ ]:
# ---- Sensor Fusion Visualization ----
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Panel 1: Position comparison
ax = axes[0, 0]
ax.plot(time_fusion, true_pos, color='steelblue', linewidth=2, label='True position')
ax.plot(time_fusion, imu_only_pos, color='goldenrod', linewidth=1.5, linestyle=':',
        alpha=0.8, label='IMU only (drifts)')
ax.plot(time_fusion, gps_only_pos, color='coral', linewidth=1, alpha=0.6,
        label='GPS only (noisy)')
ax.plot(time_fusion, est_fused[:, 0], color='seagreen', linewidth=2,
        label='Fused estimate')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Position (m)')
ax.set_title('Position Estimation')
ax.legend(fontsize=9)

# Panel 2: Position error comparison
ax = axes[0, 1]
ax.plot(time_fusion, imu_only_pos - true_pos, color='goldenrod', linewidth=1.5,
        alpha=0.7, label='IMU only error')
ax.plot(time_fusion, gps_only_pos - true_pos, color='coral', linewidth=1,
        alpha=0.5, label='GPS only error')
ax.plot(time_fusion, est_fused[:, 0] - true_pos, color='seagreen', linewidth=2,
        label='Fused error')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Position Error (m)')
ax.set_title('Position Error Comparison')
ax.legend(fontsize=9)

# Panel 3: Bias estimation
ax = axes[1, 0]
ax.plot(time_fusion, imu_bias, color='steelblue', linewidth=2, label='True bias')
ax.plot(time_fusion, est_fused[:, 2], color='seagreen', linewidth=2,
        label='Estimated bias')
sigma_bias = np.sqrt(est_P_fused[:, 2, 2])
ax.fill_between(time_fusion, est_fused[:, 2] - 2*sigma_bias,
                est_fused[:, 2] + 2*sigma_bias,
                color='seagreen', alpha=0.15, label='$\\pm 2\\sigma$')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Bias (m/s$^2$)')
ax.set_title('IMU Bias Estimation')
ax.legend(fontsize=9)

# Panel 4: Velocity estimation
ax = axes[1, 1]
ax.plot(time_fusion, true_vel, color='steelblue', linewidth=2, label='True velocity')
ax.plot(time_fusion, est_fused[:, 1], color='seagreen', linewidth=2,
        label='Fused estimate')
ax.plot(time_fusion, imu_only_vel, color='goldenrod', linewidth=1.5, linestyle=':',
        alpha=0.8, label='IMU only')
sigma_vel_fus = np.sqrt(est_P_fused[:, 1, 1])
ax.fill_between(time_fusion, est_fused[:, 1] - 2*sigma_vel_fus,
                est_fused[:, 1] + 2*sigma_vel_fus,
                color='seagreen', alpha=0.15, label='$\\pm 2\\sigma$')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Velocity (m/s)')
ax.set_title('Velocity Estimation')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

---
## 9. Summary Visualizations

In [ ]:
# ---- 4-Panel Summary Figure ----
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# ---- Panel 1: Uncertainty ellipses evolving (2D localization) ----
ax = axes[0, 0]
ax.plot(true_states_2d[:, 0], true_states_2d[:, 1], color='steelblue',
        linewidth=2, label='True path')
ax.plot(est_states_2d[:, 0], est_states_2d[:, 1], color='seagreen',
        linewidth=1.5, label='KF estimate')

# Color ellipses by time
ellipse_steps = np.arange(0, N_loc, 15)
colors_time = plt.cm.plasma(np.linspace(0.1, 0.9, len(ellipse_steps)))
for idx, k in enumerate(ellipse_steps):
    plot_uncertainty_ellipse(
        ax, est_states_2d[k, :2], est_covs_2d[k, :2, :2],
        n_std=2, fill=False, edgecolor=colors_time[idx], linewidth=1.5, alpha=0.7
    )

ax.scatter(LANDMARKS[:, 0] if 'LANDMARKS' in dir() else [],
           LANDMARKS[:, 1] if 'LANDMARKS' in dir() else [],
           color='coral', s=0)  # placeholder
ax.set_xlabel('x (m)')
ax.set_ylabel('y (m)')
ax.set_title('Uncertainty Ellipses Over Time (2D Localization)')
ax.legend(fontsize=9)
ax.set_aspect('equal')

# ---- Panel 2: Innovation sequence with bounds (falling object) ----
ax = axes[0, 1]
ax.plot(time_fall, innovations_fall[:, 0], color='steelblue', linewidth=1, alpha=0.7)
sigma_innov = np.sqrt(innov_covs_fall[:, 0, 0])
ax.fill_between(time_fall, -2*sigma_innov, 2*sigma_innov,
                color='coral', alpha=0.15, label='$\\pm 2\\sigma$ bounds')
ax.axhline(0, color='black', linewidth=0.5)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Innovation (m)')
ax.set_title('Innovation Sequence (1D Falling Object)')
ax.legend(fontsize=9)

# ---- Panel 3: Estimation error with bounds (falling object) ----
ax = axes[1, 0]
pos_error = est_states[:, 0] - true_states[:, 0]
sigma_pos_err = np.sqrt(est_covs[:, 0, 0])
ax.plot(time_fall, pos_error, color='steelblue', linewidth=1.5, label='Position error')
ax.fill_between(time_fall, -2*sigma_pos_err, 2*sigma_pos_err,
                color='coral', alpha=0.2, label='$\\pm 2\\sigma$ bounds')
ax.axhline(0, color='black', linewidth=0.5)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Error (m)')
ax.set_title('Estimation Error with Confidence Bounds')
ax.legend(fontsize=9)

# ---- Panel 4: Sensor fusion comparison ----
ax = axes[1, 1]
# Use a subset of data for clarity
t_sub = time_fusion[:int(30/DT_IMU)]  # first 30 seconds
n_sub = len(t_sub)
ax.plot(t_sub, np.abs(imu_only_pos[:n_sub] - true_pos[:n_sub]),
        color='goldenrod', linewidth=1.5, alpha=0.8, label='|IMU only error|')
ax.plot(t_sub, np.abs(gps_only_pos[:n_sub] - true_pos[:n_sub]),
        color='coral', linewidth=1, alpha=0.6, label='|GPS only error|')
ax.plot(t_sub, np.abs(est_fused[:n_sub, 0] - true_pos[:n_sub]),
        color='seagreen', linewidth=2, label='|Fused error|')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Absolute Position Error (m)')
ax.set_title('Sensor Fusion: IMU vs GPS vs Fused')
ax.legend(fontsize=9)
ax.set_yscale('log')

plt.tight_layout()
plt.show()

---
## 10. Extensions and Further Reading

This notebook covered the foundational Kalman Filter and EKF. Several important extensions address the limitations of the EKF's first-order linearization.

### Unscented Kalman Filter (UKF)

Instead of linearizing the nonlinear functions, the UKF uses **sigma points** — a deterministic set of $2n+1$ sample points that capture the mean and covariance of the state distribution. These points are propagated through the exact nonlinear functions, and the output statistics are computed from the transformed points.

$$\mathcal{X}_0 = \hat{x}, \quad \mathcal{X}_i = \hat{x} \pm \sqrt{(n + \lambda) P}_{\text{col } i}$$

The UKF achieves **second-order accuracy** (vs. first-order for EKF) without computing Jacobians — making it easier to implement and often more accurate.

### Particle Filters

For highly nonlinear systems with **non-Gaussian** distributions (e.g., multi-modal beliefs in global localization), particle filters represent the posterior as a set of weighted samples:

$$P(x_k | z_{1:k}) \approx \sum_{i=1}^{N} w_k^{(i)} \, \delta(x - x_k^{(i)})$$

Particle filters can represent arbitrary distributions but scale poorly with state dimension.

### Observability Analysis

Not all states can be estimated from given measurements. The **observability matrix**

$$\mathcal{O} = \begin{bmatrix} H \\ HF \\ HF^2 \\ \vdots \\ HF^{n-1} \end{bmatrix}$$

must have rank $n$ for the system to be **observable**. If $\text{rank}(\mathcal{O}) < n$, some state components cannot be estimated regardless of filter quality. For example, with position-only measurements and a constant-velocity model, both position and velocity are observable (rank = 2) because velocity manifests as position changes over time.

### Connection to LQG Control

The **separation principle** states that for linear-Gaussian systems, the optimal controller can be designed in two independent stages:
1. **Kalman Filter** for state estimation (this notebook)
2. **LQR** (Linear Quadratic Regulator) for optimal control

Together they form the **LQG** (Linear Quadratic Gaussian) controller — the foundation of modern control theory. The remarkable result is that the certainty-equivalent control (using $\hat{x}$ from the KF as if it were the true state) is optimal.

### Key Takeaways

1. The **Kalman Filter** is the optimal linear estimator for Gaussian systems — it computes the exact posterior
2. The **predict-update cycle** mirrors Bayesian reasoning: prior propagation + measurement fusion
3. **Innovation analysis** (NIS test) provides a built-in consistency check for filter tuning
4. The **EKF** extends the KF to nonlinear systems via Jacobian linearization
5. **Sensor fusion** combines complementary sensors (fast+drifty IMU + slow+accurate GPS) for dramatically better estimation than either sensor alone
6. Always **verify Jacobians** via finite differences before trusting an EKF implementation